## ETAPA 5 - ENTRENAR UN MODELO SIMPLE DE CLASIFICACIÓN

### Objetivo

En este notebook se entrenará un modelo simple de clasificación con Regresión Logística para el proyecto RiskScore OSCE.

El objetivo es predecir la variable `target_sancionado_24m`, que indica si un proveedor registra una sanción posterior dentro de los 24 meses.

### Regla de trabajo

El archivo featured no será modificado directamente. Se leerá el archivo, se prepararán variables predictoras, se definirá la variable objetivo y se entrenará un modelo inicial de clasificación.

In [1]:
# ==========================================
# FASE 5
# ENTRENAR MODELO SIMPLE DE CLASIFICACIÓN
# RISK SCORE OSCE
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ============================================================
# 1. Definir ruta del archivo de entrada
# ============================================================

ruta_featured = r"C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_FEATURED.csv"

# ============================================================
# 2. Cargar el dataset con features
# IMPORTANTE: el archivo usa punto y coma como separador
# ============================================================

df_featured = pd.read_csv(ruta_featured, sep=";")

# ============================================================
# 3. Crear una copia de trabajo
# ============================================================

df_modelo = df_featured.copy()

# ============================================================
# 4. Mostrar información inicial
# ============================================================

print("Dimensión del dataset featured:", df_modelo.shape)

print("\nColumnas disponibles:")
print(df_modelo.columns.tolist())

print("\nPrimeras filas:")
display(df_modelo.head())

Dimensión del dataset featured: (3875, 42)

Columnas disponibles:
['RUC', 'NOMBRE_RAZONODENOMINACIONSOCIAL', 'FECHA_INICIO', 'FECHA_FIN', 'NUMERO_RESOLUCION', 'ID_MOTIVO_INFRACCION', 'DE_MOTIVO_INFRACCION', 'motivo_grupo', 'duracion_dias', 'duracion_grupo', 'ruc_prefijo', 'fecha_segunda', 'target_sancionado_24m', 'FECHA_INICIO_DT', 'FECHA_FIN_DT', 'ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'PERIODO_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'DURACION_DIAS_CORREGIDA', 'RUC_STR', 'RUC_PREFIJO', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_TEXTO', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'TOTAL_SANCIONES_PROVEEDOR', 'ORDEN_SANCION_PROVEEDOR', 'PROVEEDOR_CON_MULTIPLES_SANCIONES', 'FECHA_PRIMERA_SANCION_PROVEEDOR', 'DIAS_DESDE_PRIMERA_SANCION']

P

,RUC,NOMBRE_RAZONODENOMINACIONSOCIAL,FECHA_INICIO,FECHA_FIN,NUMERO_RESOLUCION,ID_MOTIVO_INFRACCION,DE_MOTIVO_INFRACCION,motivo_grupo,duracion_dias,duracion_grupo,...,MOTIVO_INFO_INEXACTA,MOTIVO_INCUMPLIMIENTO,MOTIVO_CONTRATAR_IMPEDIDO,NUM_MOTIVOS_INFRACCION,TIENE_MULTIPLES_MOTIVOS,TOTAL_SANCIONES_PROVEEDOR,ORDEN_SANCION_PROVEEDOR,PROVEEDOR_CON_MULTIPLES_SANCIONES,FECHA_PRIMERA_SANCION_PROVEEDOR,DIAS_DESDE_PRIMERA_SANCION
0,2029124996,G & D CORPORACION DE NEGOCIOS LACTEOS S.A. COR...,20050608,NaN,508-2005-TC-SU,8,DOCUMENTOS FALSOS,Documentación falsa/inexacta,NaN,Sin fecha fin,...,0,0,0,1,0,1,1,0,2005-06-08,0
1,10000282125,GOMEZ CASTRO JUANA,20200616,20230716.0,1110-2020-TCE-S2,"215,",Presentar documentos falsos o adulterados a la...,Documentación falsa/inexacta,1125.0,Más de 3 años,...,0,0,0,1,0,1,1,0,2020-06-16,0
2,10000710623,OLIVEIRA DE MACHUCA LITA,20191107,20230207.0,2903-2019-TCE-S2,"214,215,",Presentar información inexacta a las Entidades...,Documentación falsa/inexacta,1188.0,Más de 3 años,...,1,0,0,2,1,1,1,0,2019-11-07,0
3,10001253609,AGUIRRE VARGAS GUSTAVO ALFREDO,20190807,20220907.0,2147-2019-TCE-S3,"214,215,",Presentar información inexacta a las Entidades...,Documentación falsa/inexacta,1127.0,Más de 3 años,...,1,0,0,2,1,1,1,0,2019-08-07,0
4,10002101756,MENDOZA PENA JULIO CESAR,20231005,20240205.0,3820-2023-TCE-S6,"241,",f) Ocasionar que la Entidad resuelva el contra...,Resolución/rescisión contractual,123.0,Hasta 6 meses,...,0,0,0,1,0,1,1,0,2023-10-05,0


In [2]:
# ============================================================
# 5. Definir variable objetivo
# ============================================================

target = "target_sancionado_24m"

print("Variable objetivo:", target)

print("\nDistribución de la variable objetivo:")
print(df_modelo[target].value_counts())

print("\nDistribución porcentual:")
print(df_modelo[target].value_counts(normalize=True) * 100)

Variable objetivo: target_sancionado_24m

Distribución de la variable objetivo:
target_sancionado_24m
0    3139
1     736
Name: count, dtype: int64

Distribución porcentual:
target_sancionado_24m
0    81.006452
1    18.993548
Name: proportion, dtype: float64


In [3]:
# ============================================================
# 6. Seleccionar variables predictoras iniciales
# ============================================================

columnas_modelo = [
    # Variables temporales creadas en Fase 4
    "ANIO_INICIO_SANCION",
    "MES_INICIO_SANCION",
    "TRIMESTRE_INICIO_SANCION",
    
    # Variables de duración / severidad
    "DURACION_MESES",
    "DURACION_ANIOS",
    "SANCION_MAYOR_1_ANIO",
    "SANCION_MAYOR_3_ANIOS",
    "LOG_DURACION_DIAS",
    
    # Variables del tipo de proveedor
    "ES_PERSONA_NATURAL",
    "ES_PERSONA_JURIDICA",
    "ES_OTRO_TIPO_RUC",
    
    # Variables del motivo de infracción
    "MOTIVO_DOC_FALSA",
    "MOTIVO_INFO_INEXACTA",
    "MOTIVO_INCUMPLIMIENTO",
    "MOTIVO_CONTRATAR_IMPEDIDO",
    "NUM_MOTIVOS_INFRACCION",
    "TIENE_MULTIPLES_MOTIVOS",
    
    # Variables categóricas originales o agrupadas
    "motivo_grupo",
    "duracion_grupo"
]

# Verificar que las columnas existan
columnas_existentes = [col for col in columnas_modelo if col in df_modelo.columns]
columnas_faltantes = [col for col in columnas_modelo if col not in df_modelo.columns]

print("Columnas que sí existen y se usarán:")
print(columnas_existentes)

print("\nColumnas faltantes, no se usarán:")
print(columnas_faltantes)

# Quedarnos solo con columnas existentes + target
df_modelo = df_modelo[columnas_existentes + [target]].copy()

print("\nDimensión del dataset para modelamiento:")
print(df_modelo.shape)

display(df_modelo.head())

Columnas que sí existen y se usarán:
['ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'motivo_grupo', 'duracion_grupo']

Columnas faltantes, no se usarán:
[]

Dimensión del dataset para modelamiento:
(3875, 20)


,ANIO_INICIO_SANCION,MES_INICIO_SANCION,TRIMESTRE_INICIO_SANCION,DURACION_MESES,DURACION_ANIOS,SANCION_MAYOR_1_ANIO,SANCION_MAYOR_3_ANIOS,LOG_DURACION_DIAS,ES_PERSONA_NATURAL,ES_PERSONA_JURIDICA,ES_OTRO_TIPO_RUC,MOTIVO_DOC_FALSA,MOTIVO_INFO_INEXACTA,MOTIVO_INCUMPLIMIENTO,MOTIVO_CONTRATAR_IMPEDIDO,NUM_MOTIVOS_INFRACCION,TIENE_MULTIPLES_MOTIVOS,motivo_grupo,duracion_grupo,target_sancionado_24m
0,2005,6,2,NaN,NaN,0,0,NaN,0,0,1,1,0,0,0,1,0,Documentación falsa/inexacta,Sin fecha fin,0
1,2020,6,2,37.500000,3.082192,1,1,7.026427,1,0,0,1,0,0,0,1,0,Documentación falsa/inexacta,Más de 3 años,0
2,2019,11,4,39.600000,3.254795,1,1,7.080868,1,0,0,1,1,0,0,2,1,Documentación falsa/inexacta,Más de 3 años,0
3,2019,8,3,37.566667,3.087671,1,1,7.028201,1,0,0,1,1,0,0,2,1,Documentación falsa/inexacta,Más de 3 años,0
4,2023,10,4,4.100000,0.336986,0,0,4.820282,1,0,0,0,0,0,0,1,0,Resolución/rescisión contractual,Hasta 6 meses,0


In [4]:
# ============================================================
# 7. Limpieza básica antes del modelamiento
# ============================================================

# Reemplazar infinitos por NaN
df_modelo = df_modelo.replace([np.inf, -np.inf], np.nan)

# Mostrar cantidad de nulos por columna
print("Nulos antes del tratamiento:")
print(df_modelo.isnull().sum())

# Separar columnas numéricas y categóricas
columnas_numericas = df_modelo.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = df_modelo.select_dtypes(include=["object"]).columns.tolist()

# Quitar target de las numéricas si aparece allí
if target in columnas_numericas:
    columnas_numericas.remove(target)

# Rellenar nulos numéricos con la mediana
for col in columnas_numericas:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())

# Rellenar nulos categóricos con "SIN_DATO"
for col in columnas_categoricas:
    df_modelo[col] = df_modelo[col].fillna("SIN_DATO")

print("\nNulos después del tratamiento:")
print(df_modelo.isnull().sum())

Nulos antes del tratamiento:
ANIO_INICIO_SANCION            0
MES_INICIO_SANCION             0
TRIMESTRE_INICIO_SANCION       0
DURACION_MESES               528
DURACION_ANIOS               528
SANCION_MAYOR_1_ANIO           0
SANCION_MAYOR_3_ANIOS          0
LOG_DURACION_DIAS            539
ES_PERSONA_NATURAL             0
ES_PERSONA_JURIDICA            0
ES_OTRO_TIPO_RUC               0
MOTIVO_DOC_FALSA               0
MOTIVO_INFO_INEXACTA           0
MOTIVO_INCUMPLIMIENTO          0
MOTIVO_CONTRATAR_IMPEDIDO      0
NUM_MOTIVOS_INFRACCION         0
TIENE_MULTIPLES_MOTIVOS        0
motivo_grupo                   0
duracion_grupo                 0
target_sancionado_24m          0
dtype: int64

Nulos después del tratamiento:
ANIO_INICIO_SANCION          0
MES_INICIO_SANCION           0
TRIMESTRE_INICIO_SANCION     0
DURACION_MESES               0
DURACION_ANIOS               0
SANCION_MAYOR_1_ANIO         0
SANCION_MAYOR_3_ANIOS        0
LOG_DURACION_DIAS            0
ES_PERSONA_NATURAL

In [5]:
# ============================================================
# 8. Separar variables predictoras (X) y variable objetivo (y)
# ============================================================

X = df_modelo.drop(columns=[target])
y = df_modelo[target]

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)

print("\nDistribución del target:")
print(y.value_counts())

Dimensión de X: (3875, 19)
Dimensión de y: (3875,)

Distribución del target:
target_sancionado_24m
0    3139
1     736
Name: count, dtype: int64


In [6]:
# ============================================================
# 9. Convertir variables categóricas a variables numéricas
# ============================================================

X = pd.get_dummies(X, drop_first=True)

print("Dimensión de X después de get_dummies:")
print(X.shape)

print("\nPrimeras columnas resultantes:")
print(X.columns.tolist()[:30])

Dimensión de X después de get_dummies:
(3875, 26)

Primeras columnas resultantes:
['ANIO_INICIO_SANCION', 'MES_INICIO_SANCION', 'TRIMESTRE_INICIO_SANCION', 'DURACION_MESES', 'DURACION_ANIOS', 'SANCION_MAYOR_1_ANIO', 'SANCION_MAYOR_3_ANIOS', 'LOG_DURACION_DIAS', 'ES_PERSONA_NATURAL', 'ES_PERSONA_JURIDICA', 'ES_OTRO_TIPO_RUC', 'MOTIVO_DOC_FALSA', 'MOTIVO_INFO_INEXACTA', 'MOTIVO_INCUMPLIMIENTO', 'MOTIVO_CONTRATAR_IMPEDIDO', 'NUM_MOTIVOS_INFRACCION', 'TIENE_MULTIPLES_MOTIVOS', 'motivo_grupo_Impedimento para contratar', 'motivo_grupo_No perfecciona/suscribe contrato', 'motivo_grupo_Otros', 'motivo_grupo_Resolución/rescisión contractual', 'duracion_grupo_6-12 meses', 'duracion_grupo_Fecha fin < inicio', 'duracion_grupo_Hasta 6 meses', 'duracion_grupo_Más de 3 años', 'duracion_grupo_Sin fecha fin']


In [7]:
# ============================================================
# 10. Dividir en conjunto de entrenamiento y prueba
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dimensión de X_train:", X_train.shape)
print("Dimensión de X_test:", X_test.shape)
print("Dimensión de y_train:", y_train.shape)
print("Dimensión de y_test:", y_test.shape)

print("\nDistribución en y_train:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en y_test:")
print(y_test.value_counts(normalize=True) * 100)

Dimensión de X_train: (3100, 26)
Dimensión de X_test: (775, 26)
Dimensión de y_train: (3100,)
Dimensión de y_test: (775,)

Distribución en y_train:
target_sancionado_24m
0    81.0
1    19.0
Name: proportion, dtype: float64

Distribución en y_test:
target_sancionado_24m
0    81.032258
1    18.967742
Name: proportion, dtype: float64


In [8]:
# ============================================================
# 11. Crear el modelo de Regresión Logística
# ============================================================

modelo = LogisticRegression(
    max_iter=20000,
    random_state=42,
    class_weight="balanced"
)

print("Modelo creado:")
print(modelo)

# ============================================================
# 12. Entrenar el modelo
# ============================================================

modelo.fit(X_train, y_train)

print("\nModelo entrenado correctamente.")
print("Clases detectadas por el modelo:")
print(modelo.classes_)

Modelo creado:
LogisticRegression(class_weight='balanced', max_iter=20000, random_state=42)

Modelo entrenado correctamente.
Clases detectadas por el modelo:
[0 1]


In [9]:
class_weight="balanced"

In [10]:
# ============================================================
# 13. Generar predicciones iniciales
# ============================================================

y_pred = modelo.predict(X_test)

# Probabilidad de pertenecer a la clase positiva: 1
y_prob = modelo.predict_proba(X_test)[:, 1]

print("Primeras 10 predicciones:")
print(y_pred[:10])

print("\nPrimeras 10 probabilidades de sanción futura:")
print(y_prob[:10])

Primeras 10 predicciones:
[0 1 0 0 1 0 1 1 0 1]

Primeras 10 probabilidades de sanción futura:
[0.4480394  0.63929793 0.41292219 0.45208491 0.60737889 0.38812931
 0.5796798  0.52096224 0.41661818 0.53530039]


In [11]:
# ============================================================
# 14. Resultados iniciales del modelo
# ============================================================

print("Accuracy inicial:")
print(accuracy_score(y_test, y_pred))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

Accuracy inicial:
0.7135483870967742

Matriz de confusión:
[[483 145]
 [ 77  70]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.86      0.77      0.81       628
           1       0.33      0.48      0.39       147

    accuracy                           0.71       775
   macro avg       0.59      0.62      0.60       775
weighted avg       0.76      0.71      0.73       775



In [12]:
# ============================================================
# 15. Crear tabla de resultados con RiskScore
# ============================================================

resultados = X_test.copy()

resultados["target_real"] = y_test.values
resultados["prediccion_modelo"] = y_pred
resultados["probabilidad_sancion_24m"] = y_prob

# Crear niveles de riesgo
resultados["nivel_riesgo"] = pd.cut(
    resultados["probabilidad_sancion_24m"],
    bins=[0, 0.30, 0.60, 1.00],
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True
)

print("Primeros resultados con score de riesgo:")
display(resultados[[
    "target_real",
    "prediccion_modelo",
    "probabilidad_sancion_24m",
    "nivel_riesgo"
]].head(20))

print("\nDistribución por nivel de riesgo:")
print(resultados["nivel_riesgo"].value_counts())

Primeros resultados con score de riesgo:


,target_real,prediccion_modelo,probabilidad_sancion_24m,nivel_riesgo
1220,0,0,0.448039,Medio
3825,1,1,0.639298,Alto
1770,0,0,0.412922,Medio
3393,0,0,0.452085,Medio
2900,0,1,0.607379,Alto
253,0,0,0.388129,Medio
656,0,1,0.579680,Medio
3209,0,1,0.520962,Medio
2575,0,0,0.416618,Medio
3154,0,1,0.535300,Medio



Distribución por nivel de riesgo:
nivel_riesgo
Medio    668
Alto      90
Bajo      17
Name: count, dtype: int64


In [13]:
# ============================================================
# 16. Guardar resultados de predicción
# ============================================================

ruta_predicciones = r"C:\Proyecto-Risk-Score-OECE\data\processed\predicciones_riskscore_osce.csv"

resultados.to_csv(
    ruta_predicciones,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo de predicciones guardado en:")
print(ruta_predicciones)

Archivo de predicciones guardado en:
C:\Proyecto-Risk-Score-OECE\data\processed\predicciones_riskscore_osce.csv


In [14]:
# ============================================================
# 15. CREAR TABLA DE RESULTADOS CON RUC Y NOMBRE DEL PROVEEDOR
# ============================================================

# Recuperamos las filas originales que quedaron en el conjunto de prueba
# X_test conserva los índices originales del dataframe
indices_test = X_test.index

# Creamos una tabla con datos identificadores del proveedor
resultados = df_featured.loc[indices_test, [
    "RUC",
    "NOMBRE_RAZONODENOMINACIONSOCIAL"
]].copy()

# Agregamos el valor real, la predicción y la probabilidad
resultados["target_real"] = y_test.values
resultados["prediccion_modelo"] = y_pred
resultados["probabilidad_sancion_24m"] = y_prob

# Creamos niveles de riesgo
resultados["nivel_riesgo"] = pd.cut(
    resultados["probabilidad_sancion_24m"],
    bins=[0, 0.30, 0.60, 1.00],
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True
)

# Ordenamos de mayor a menor riesgo
resultados = resultados.sort_values(
    by="probabilidad_sancion_24m",
    ascending=False
)

print("Resultados con riesgo atribuido a cada proveedor:")
display(resultados.head(20))

Resultados con riesgo atribuido a cada proveedor:


,RUC,NOMBRE_RAZONODENOMINACIONSOCIAL,target_real,prediccion_modelo,probabilidad_sancion_24m,nivel_riesgo
1731,20489091675,SISA TOURS SAC,0,1,0.833974,Alto
2325,20527015627,SERVICENTRO EL PORVENIR E.I.R.L. - SERVICENTRO...,1,1,0.832107,Alto
3303,20600738179,PERUANA DE SERVICIOS INTEGRALES S.A.C.,1,1,0.828399,Alto
1528,20480052090,NEGOCIOS & CONSTRUCCIONES LITO E.I.R.L.,0,1,0.826262,Alto
3194,20600192613,FERSCONS E.I.R.L.,1,1,0.825499,Alto
2441,20531497156,G&M NETWORK E.I.R.L.,1,1,0.818459,Alto
1077,20352428648,J.P.C. INGENIEROS S.A.C.,0,1,0.817975,Alto
2438,20531345305,RADIO MASTER EIRL,0,1,0.817413,Alto
1234,20414235108,SCHNEIDER ELECTRIC SYSTEMS DEL PERU S.A.,0,1,0.816951,Alto
123,10086163662,ESPINOZA RODAS GUILLERMO,0,1,0.816910,Alto


In [15]:
# ============================================================
# 16. Guardar resultados de predicción
# ============================================================

ruta_predicciones = r"C:\Proyecto-Risk-Score-OECE\data\processed\predicciones_riskscore_osce.csv"

resultados.to_csv(
    ruta_predicciones,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo de predicciones guardado en:")
print(ruta_predicciones)

Archivo de predicciones guardado en:
C:\Proyecto-Risk-Score-OECE\data\processed\predicciones_riskscore_osce.csv
